<div style="background-color: #f8fafc; border-left: 6px solid #e11d48; padding: 20px 24px; border-radius: 6px; box-shadow: 0 1px 3px rgba(0,0,0,0.06); font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Arial, sans-serif;">
  <span style="font-size: 11px; font-weight: 700; text-transform: uppercase; letter-spacing: 1.5px; color: #e11d48;">AIML Honors &bull; Lab Assignment 05</span>
  <h1 style="margin: 8px 0 12px 0; color: #0f172a; font-size: 24px; font-weight: 700;">Python Metaprogramming & Custom Function Decorators</h1>
  <div style="font-size: 14px; color: #475569; line-height: 1.6;">
    <strong>Student Name:</strong> Pranjal Khadke &nbsp;|&nbsp; 
    <strong>Email:</strong> <a href="mailto:Pranjalckhadke@gmail.com" style="color: #e11d48; text-decoration: none;">Pranjalckhadke@gmail.com</a> &nbsp;|&nbsp; 
    <strong>GitHub:</strong> <a href="https://github.com/PranjalKhadke" style="color: #e11d48; text-decoration: none;">github.com/PranjalKhadke</a><br>
    <strong>Course:</strong> AIML Honors &bull; Sem III &nbsp;|&nbsp; 
    <strong>Topic:</strong> Higher-Order Functions, Closures, Introspection & Decorator Stacking
  </div>
</div>

### 1. Aim & Theoretical Foundations

**Aim:** To engineer custom Python decorators demonstrating metaprogramming paradigms, parameter introspection, execution performance benchmarking using high-precision hardware timers, input security guards, and multi-tier decorator chaining.

#### Core Metaprogramming Concepts:
1. **First-Class Functions & Lexical Closures:** In Python, functions are first-class citizens. A decorator is a higher-order function that captures an inner function inside its lexical closure, enriching behavior without modifying original function source code.
2. **Preserving Function Identity (`functools.wraps`):** Applying `@wraps(func)` copies `__name__`, `__doc__`, and annotation metadata from the target callable to the inner wrapper, preventing introspection breakdown.
3. **Decorator Stacking Pipeline:** Chaining multiple decorators evaluates from the inside-out (bottom-to-top), while executing from the outside-in (top-to-bottom):
   $$\text{Decorated } f = D_1(D_2(D_3(f)))$$

#### Architecture of Implemented Decorators:
| Decorator | Parameters | Purpose | Scope |
| :--- | :--- | :--- | :--- |
| **`@time_execution`** | None | Measures runtime in milliseconds using `time.perf_counter()` | Performance profiling |
| **`@log_calls`** | `tag: str` | Intercepts positional (`*args`) & keyword (`**kwargs`) inputs + return value | Audit logging |
| **`@validate_positive`**| None | Validates numeric inputs are strictly positive ($> 0$) | Security & validation |


In [1]:
import time
from functools import wraps

def time_execution(func):
    """Decorator to benchmark execution runtime with microsecond precision."""
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed = time.perf_counter() - start
        print(f"  [PROFILER] {func.__name__}() elapsed time: {elapsed * 1000:.4f} ms ({elapsed:.7f} sec)")
        return result
    return wrapper

In [2]:
def log_calls(tag=None):
    """Parameterized decorator logging function invocations, arguments, and return values."""
    def decorator(func):
        label = tag if tag else func.__name__.upper()
        
        @wraps(func)
        def wrapper(*args, **kwargs):
            arg_list = [repr(a) for a in args]
            kwarg_list = [f"{k}={repr(v)}" for k, v in kwargs.items()]
            call_repr = ", ".join(arg_list + kwarg_list)
            
            print(f"  [{label}] Calling {func.__name__}({call_repr})")
            result = func(*args, **kwargs)
            print(f"  [{label}] Returned -> {result}")
            return result
        return wrapper
    return decorator

In [3]:
def validate_positive(func):
    """Validation guard ensuring all positional numeric arguments are strictly positive (> 0)."""
    @wraps(func)
    def wrapper(*args, **kwargs):
        for idx, arg in enumerate(args):
            if isinstance(arg, (int, float)) and arg <= 0:
                raise ValueError(f"Positional argument at index {idx} ({arg}) must be strictly positive (> 0).")
        return func(*args, **kwargs)
    return wrapper

In [4]:
# Test Case 1: Parameter & return value introspection
@log_calls(tag="ALGEBRA")
def compute_power(base, exponent):
    return base ** exponent

res = compute_power(2, 10)
print(f"Computed Result: {res}")

  [ALGEBRA] Calling compute_power(2, 10)
  [ALGEBRA] Returned -> 1024
Computed Result: 1024


In [5]:
# Test Case 2: High-precision execution benchmarking
@time_execution
def sum_of_squares(n):
    return sum(i * i for i in range(1, n + 1))

total = sum_of_squares(100000)
print(f"Sum of First 100,000 Squares: {total:,}")

  [PROFILER] sum_of_squares() elapsed time: 3.8210 ms (0.0038210 sec)
Sum of First 100,000 Squares: 333,338,333,350,000


In [6]:
# Test Case 3: Chaining multi-tier decorators (@validate_positive + @log_calls + @time_execution)
@validate_positive
@log_calls(tag="FINANCE-ENGINE")
@time_execution
def calculate_compound_interest(principal, rate_pct, years):
    amount = principal * ((1 + rate_pct / 100) ** years)
    return round(amount - principal, 2)

interest = calculate_compound_interest(50000, 7.5, 3)
print(f"Accrued Compound Interest: Rs. {interest:,.2f}")

  [FINANCE-ENGINE] Calling calculate_compound_interest(50000, 7.5, 3)
  [PROFILER] calculate_compound_interest() elapsed time: 0.0070 ms (0.0000070 sec)
  [FINANCE-ENGINE] Returned -> 12114.84
Accrued Compound Interest: Rs. 12,114.84


In [7]:
# Test Case 4: Security guard rail intercepting invalid negative principal
try:
    calculate_compound_interest(-25000, 6.0, 2)
except ValueError as err:
    print(f"  [INTERCEPTED] Security Guard Active: {err}")

  [INTERCEPTED] Security Guard Active: Positional argument at index 0 (-25000) must be strictly positive (> 0).


<div style="background-color: #f0fdf4; border: 1px solid #bbf7d0; border-left: 5px solid #16a34a; padding: 16px 20px; border-radius: 6px; font-family: sans-serif;">
  <h4 style="margin: 0 0 6px 0; color: #15803d; font-size: 15px; font-weight: 700;">Conclusion & Engineering Summary</h4>
  <p style="margin: 0; color: #166534; font-size: 13.5px; line-height: 1.6;">
    Custom Python decorators proved to be an invaluable design pattern for non-invasive cross-cutting concerns (benchmarking, logging, validation). Chaining <code>@validate_positive</code>, <code>@log_calls</code>, and <code>@time_execution</code> successfully enforced security invariants before executing mathematical and financial computations, all while keeping business logic clean and isolated.
  </p>
</div>